# Competitors # 

## Read in dataset ##

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

import resicon
from resicon import *

import geostas
import mdtraj as md
import MDAnalysis as mda 
from redpandda import *
import time

import redpandda_without_ship
# import visualizations
import comodo

In [ ]:
folder = "McGibbon/"
trajectory_file = 'trajectory-3.xtc'
peptide_file = "fs-peptide.pdb"
filename = "fs-peptide_trajectory-3"
output_filename = "data/fspeptide_traj3.csv"
full_folder = "trajdata/"+ folder
multifile_name = full_folder + "trajectory-" + "_multipdb.pdb"
# traj = md.load(full_folder+trajectory_file, top=full_folder+peptide_file)
# traj.save_pdb(multifile_name)
ship_file = "data/ship_results_fspeptide_3.csv"


In [ ]:
md_trajectory_info = ['prod_r1_nojump_prot.xtc','prod_r1_pbc_fit_prot_last.pdb','proteins_comet/ProtNo2/',None,None]

folder = 'proteins_comet/ProtNo2/'
trajectory_file = 'prod_r1_nojump_prot.xtc'
peptide_file = "prod_r1_pbc_fit_prot_last.pdb"
filename = "tll_trajectory"
output_filename = "data/tll_protein.csv"
full_folder = "trajdata/"+ folder
multifile_name = full_folder + "trajectory-" + "_multipdb.pdb"
ship_file = "data/ship_results_tll.csv"

In [ ]:
# folder = "HIV1Protease/"
# trajectory_file = '1hhp.dcd'
# peptide_file = "1hhp.pdb"
# filename = "hiv_1"
# output_filename = "hiv_1.csv"
# full_folder = "trajdata/"+ folder
# multifile_name = full_folder + "trajectory-" + "_multipdb.pdb"
# traj = md.load(full_folder+trajectory_file, top=full_folder+peptide_file)
# traj.save_pdb(multifile_name)
# ship_file = "ship_hiv_1.csv"


## Prelim ##

In [ ]:
def compute_dist_matrices(traj_array):
    # create matrices for plotting
    dist_matrices = redpandda_general.get_distance_matrices(traj_array)
    delta_matrices = redpandda_general.get_delta_matrices(dist_matrices)
    delta_matrices_wo_absolute = redpandda_general.get_delta_matrices_wo_absolute(dist_matrices)


    average_delta_matrix = redpandda_general.calculate_average_delta_matrix(delta_matrices)
    average_delta_matrix_wo_absolute = redpandda_general.calculate_average_delta_matrix(delta_matrices_wo_absolute)

    average_distance_matrix = redpandda_general.calculate_average_delta_matrix(dist_matrices)

    std_distance_matrix = redpandda_general.get_std_matrices(dist_matrices)
    std_delta_matrix = redpandda_general.get_std_matrices(delta_matrices)

    stddv_matrices = redpandda_general.get_stddv(dist_matrices)


    summed_delta_matrix_1std = average_delta_matrix + std_delta_matrix 
    summed_delta_matrix_2std = average_delta_matrix + std_delta_matrix * 2
    return(dist_matrices)

In [ ]:
def preprocess_protein_trajectory(prot_info, k_cluster=None):
  frames_count = prot_info[3]
  traj_array, k_cluster = preprocessing(prot_info,frames_count,k_cluster)
  return traj_array, k_cluster

In [ ]:
md_trajectory_info = [trajectory_file,peptide_file,folder,None,None]

In [ ]:
#how many frames to process
frames_count = md_trajectory_info[3]

trajectory_file = md_trajectory_info[0].split()[0]
pdb_file = md_trajectory_info[1].split()[0]

In [ ]:
# call MD-related preprocessing
traj_array, k_cluster = preprocess_protein_trajectory(md_trajectory_info)

In [ ]:
def compute_dist_matrices_without_ship(traj_array):
    # create matrices for plotting
    dist_matrices = redpandda_without_ship.get_distance_matrices(traj_array)
    delta_matrices = redpandda_without_ship.get_delta_matrices(dist_matrices)
    delta_matrices_wo_absolute = redpandda_without_ship.get_delta_matrices_wo_absolute(dist_matrices)


    average_delta_matrix = redpandda_without_ship.calculate_average_delta_matrix(delta_matrices)
    average_delta_matrix_wo_absolute = redpandda_without_ship.calculate_average_delta_matrix(delta_matrices_wo_absolute)

    average_distance_matrix = redpandda_without_ship.calculate_average_delta_matrix(dist_matrices)

    std_distance_matrix = redpandda_without_ship.get_std_matrices(dist_matrices)
    std_delta_matrix = redpandda_without_ship.get_std_matrices(delta_matrices)

    stddv_matrices = redpandda_without_ship.get_stddv(dist_matrices)


    summed_delta_matrix_1std = average_delta_matrix + std_delta_matrix 
    summed_delta_matrix_2std = average_delta_matrix + std_delta_matrix * 2
    return(dist_matrices)

## Resicon ##

In [ ]:
resi_dict = dict()

start_time = time.time()

md_trajectory_info = [trajectory_file,peptide_file,folder,None,None]
resi_clustering = resicon(trajectory_file,peptide_file,folder,k_cluster, only_CA=True)

curr_time = time.time() - start_time 
resi_dict["runtime"] = curr_time
resi_dict["dataset"] = filename
    
resi_dict["name"] = "Resicon"
resi_dict["clustering"] = np.array(resi_clustering)
dist_matrices = compute_dist_matrices_without_ship(traj_array)

Q, _ = cc.get_Q_for_clustering(dist_matrices, resi_dict["clustering"], k=len(np.unique(resi_clustering)))
resi_dict["Q"] = Q

## COMODO ##

In [ ]:
comodo_dict = dict()

start_time = time.time()

com_pd_eln = comodo.full_comodo_clustering((full_folder+peptide_file), elastic_network=True)
curr_time = time.time() - start_time 
comodo_dict["runtime"] = curr_time

comodo_dict["name"] = "Comodo"
comodo_dict["method"] = "Comodo"
comodo_dict["params"] = "Elastic Network"
comodo_dict["clustering"] = np.array(com_pd_eln)
Q, _ = cc.get_Q_for_clustering(dist_matrices, comodo_dict["clustering"], k=len(np.unique(com_pd_eln)))
comodo_dict["Q"] = Q


## GeoStas ## 

In [ ]:
from geostas import *
import time
import os
geostas_dict = dict()
start_time = time.time()


import os
import numpy as np
import time
from geostas import compute_geostas_clusters_multipdb

geostas_clusters = compute_geostas_clusters_multipdb(
    filename=multifile_name,
    only_CA=True,
    k=k_cluster
)

curr_time = time.time() - start_time 
geostas_dict["runtime"] = curr_time

geostas_dict["name"] = "Geostas"
geostas_dict["method"] = "Geostas"
geostas_dict["params"] = ""
geostas_dict["clustering"] = np.array(geostas_clusters)
Q, _ = cc.get_Q_for_clustering(dist_matrices, geostas_dict["clustering"], k=len(np.unique(geostas_clusters)))
geostas_dict["Q"] = Q

## Combine results df ##

In [ ]:
df_results = pd.read_csv(ship_file)

arr = np.fromstring(df_results["clustering"].to_list()[1].strip('[]'), sep=' ', dtype=int)
row = df_results.loc[1]  # row with index 0
ship_dict =  row.to_dict()

score,_ = cc.get_Q_for_clustering(dist_matrices,arr, k_cluster)
ship_dict["Q"] = score
# Combine them into a list
dicts = [ship_dict, geostas_dict, resi_dict,comodo_dict]

df = pd.DataFrame(dicts)

df.to_csv(output_filename)

In [ ]:
df

In [ ]:
print("test")

# Visualisation #

In [ ]:
import seaborn as sns

df = pd.read_csv(output_filename)
print(output_filename)
# Plot Q values
plt.figure(figsize=(10, 5))
sns.barplot(x="name", y="Q", data=df, palette="viridis")

# Labels and title
plt.xlabel("Method (Name, Matrix)")
plt.ylabel("Q Value")
plt.title("Q Values Comparison: "+filename)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig("figs/q_score"+filename+".png", dpi=300)
plt.show()

In [ ]:
import pandas as pd

# List of CSV files
csv_files = [
    "data/fspeptide_traj1.csv",
    "data/fspeptide_traj2.csv",
    "data/fspeptide_traj3.csv",
    "data/hiv_1.csv"
]

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    
    # Fill the dataset name for all rows
    # Assumes column name is 'dataset'
    df['dataset'] = df['dataset'].ffill().bfill()
    
    dfs.append(df)

# Combine all DataFrames
combined_df = pd.concat(dfs, ignore_index=True)
combined_df.to_csv("data/all_md.csv")


In [ ]:
import pandas as pd

In [ ]:
combined_df[["name","runtime","Q","dataset"]]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.boxplot(
    data=combined_df,
    x="name",
    y="Q",      # replace with your metric column
    palette="Set2"
)

plt.ylabel("Score")
plt.title("Algorithm Performance Across MD Datasets")
plt.savefig("figs/q_score_all_datasets.png", dpi=300)

plt.grid(axis="y")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.boxplot(
    data=combined_df,
    x="name",
    y="runtime",      # replace with your metric column
    palette="Set2"
)

plt.ylabel("Score")
plt.title("Algorithm Runtime Performance Across MD Datasets")
plt.savefig("figs/runtime_all_datasets.png", dpi=300)

plt.grid(axis="y")
plt.show()
